# Quantum Genome Detective · the race engine

**Owner:** Tudor · **Used by:** `tudor_classical_vs_quantum.ipynb` and each teammate's `bench/<name>.ipynb`

This notebook runs **no experiments**. It defines a fair race between QAOA and classical methods on phasing puzzles of any size, and every piece of that race is explained next to its code:

| Part | What you get |
|---|---|
| 1 · Sockets | slots where teammates plug in their own evidence, circuit, trainer, post-processing or simulator |
| 2 · Puzzles | the notebook's 5-SNP toy, plus a generator of fake puzzles of any size with a known answer |
| 3 · Cost | reads → votes `A`, `D` → couplings `J` and `offset` (same formulas as the baseline notebook) |
| 4 · Referee | the exact best cost: brute force up to 24 SNPs, band dynamic programming beyond |
| 5 · Classical | random guessing, greedy + local search, simulated annealing |
| 6 · QAOA | the baseline circuit, an exact numpy simulator, zero-shot training, Aer statevector and MPS |
| 7 · Grading | `P_budget`, `TTS99`, `score`, switch errors |
| 8 · Race | `Config`, `run_instance`, `run_bench`, `summarize`, `plot` |
| 9 · Checks | proof that the pieces agree with Qiskit, brute force and the baseline notebook |

**Use it from another notebook in this folder:**
```python
QGD_QUIET = True        # skip this notebook's small examples and checks
%run qgd_bench.ipynb    # every function below now exists in your notebook
```
Opened on its own, it also prints small examples and runs the checks (tens of seconds on a laptop).

**Environment.** Keep these notebooks together in `bench/` and use that folder as the kernel's working directory. Use a Python 3 kernel with NumPy, SciPy, pandas, matplotlib, Qiskit and Qiskit Aer installed. If needed, run `%pip install numpy scipy pandas matplotlib qiskit qiskit-aer` once in that kernel, then restart it before **Run All**. No IBM or Kaggle credentials are needed for the local experiments.

### Conventions (same as `bench/all_together.ipynb`)

| Thing | Convention |
|---|---|
| a read | one row with a value per SNP: `0`, `1`, or `-1` = this read has no letter here |
| an answer `x` | one row of 0 / 1, **SNP 1 first**. `x` is copy A, `1 - x` is copy B |
| spin `z` | `z = 1 - 2x`, so letter 0 → +1 and letter 1 → −1 |
| votes | `A[i, j]` = reads saying SNPs i and j match, `D[i, j]` = reads saying they differ (only `i < j` is filled) |
| cost | $\text{cost}(x) = \text{offset} + \sum_{i<j} J_{ij} z_i z_j$ with $J = (D - A)/2$ and $\text{offset} = \sum (A + D)/2$ |
| Qiskit keys | reversed: the rightmost character is qubit 0 = SNP 1 (`decode_key` handles it) |
| QAOA angles | one flat list `[γ1 … γp, β1 … βp]` |

In [1]:
import math
import os
import time
import zlib
from dataclasses import dataclass, field, replace

import numpy as np

QGD_QUIET = globals().get("QGD_QUIET", False)   # True when another notebook %runs this one

## 1 · Sockets: how teammates plug in

The race is built from five swappable parts. Each part is a **socket**: a named slot in `REGISTRY` where any function with the right inputs and outputs can sit.

| Socket | Your function gets | It returns | Ideas | Owner |
|---|---|---|---|---|
| `evidence` | reads | `(A, D)` | weighted votes, read quality | Alexandra |
| `qaoa` | `(J, p)` | `(circuit, θ, scale)` | warm start, multi-angle, other mixers | Carla |
| `trainer` | `(J, offset, p, circuit name, ctx)` | `(angles, training shots[, extra columns])` | CVaR, shot-based COBYLA | Carla |
| `postprocess` | `(samples, J, offset)` | `(samples, extra cost checks)` | fold mirror twins, 1-flip polish | Alex |
| `backend` | `(job, shots, seed)` | `(samples, probabilities or None)` | FakeManila, real IBM device | Alex |

Writing `@register("qaoa", "carla_ws")` above a function stores it in `REGISTRY["qaoa"]["carla_ws"]`. The race then picks it by name with `Config(qaoa="carla_ws")`. Nothing else in the engine changes. The built-in parts are registered the same way: `pairwise`, `baseline`, `cobyla`, `numpy`, `aer_sv`, `aer_mps` and `none`.

Loading the engine a second time keeps everything already in `REGISTRY`. That is what lets a merge notebook load several teammates' notebooks one after another (each one loads the engine again) and keep all their plug-ins.

In [2]:
REGISTRY: dict[str, dict] = globals().get("REGISTRY") or {
    "evidence": {}, "qaoa": {}, "trainer": {}, "postprocess": {}, "backend": {}}


def register(kind, name):
    def deco(fn):
        REGISTRY[kind][name] = fn
        return fn
    return deco

## 2 · Puzzles

### 2.1 What a puzzle is

An `Instance` holds the reads (a matrix) and, for fake puzzles, the hidden answer `truth`. **The truth is only used for grading.** No solver ever sees it.

`TOY_READS` is the baseline notebook's 5-SNP puzzle, copied exactly, so every number can be compared with the notebook.

In [3]:
@dataclass
class Instance:
    name: str
    reads: np.ndarray                 # (R, N): 0 / 1, or -1 where the read has no letter
    truth: np.ndarray | None = None   # hidden copy A; only used for reporting, never for solving
    params: dict = field(default_factory=dict)

    @property
    def n(self):
        return self.reads.shape[1]


TOY_READS = np.array([[0, 1, 1, -1, -1], [1, 0, 0, -1, -1], [-1, 1, 1, 0, -1], [-1, 0, 0, 1, -1],
                      [-1, -1, 1, 0, 1], [-1, -1, 0, 1, 0], [0, -1, -1, 0, 1], [1, -1, -1, 1, 0],
                      [0, 0, -1, -1, -1]])


def toy_instance():
    return Instance("toy5", TOY_READS.copy(), np.array([0, 1, 1, 0, 1]), {"source": "notebook"})


def show_reads(reads, truth=None):
    """Print reads as a grid: 0 / 1, '.' where the read has no letter."""
    for r, row in enumerate(np.asarray(reads)):
        print(f"read {r + 1:>2}   " + " ".join("." if v < 0 else str(v) for v in row))
    if truth is not None:
        print("truth     " + " ".join(map(str, truth)) + "   <- copy A (copy B is its mirror)")


if not QGD_QUIET:
    show_reads(TOY_READS, toy_instance().truth)

read  1   0 1 1 . .
read  2   1 0 0 . .
read  3   . 1 1 0 .
read  4   . 0 0 1 .
read  5   . . 1 0 1
read  6   . . 0 1 0
read  7   0 . . 0 1
read  8   1 . . 1 0
read  9   0 0 . . .
truth     0 1 1 0 1   <- copy A (copy B is its mirror)


### 2.2 Fake puzzles of any size, with a known answer

The notebook has a single 5-SNP puzzle. To see how methods **scale**, we need many puzzles of every size, and we must know the right answer to grade them. `make_instance` builds them the way a sequencer would:

1. **Hidden answer.** Pick `truth`, N random letters (copy A). Copy B is its mirror `1 - truth`.
2. **How many reads.** Enough to cover each SNP about `coverage` times: ⌈coverage · N / average read length⌉.
3. **Each read.** Random length within `read_len` (2–4 SNPs by default) and a random start. A coin flip picks copy A or B, and the read copies that copy's letters.
4. **Realistic mistakes (optional).** Each letter is flipped with probability `error` (sequencing error) and hidden with probability `missing` (letter not read).
5. **Gaps (optional).** `gaps` cut points that no read may cross. The pieces on either side can then flip independently, and nobody can phase them.
6. **Repair.** If two neighbouring SNPs inside a piece share no read, add a 2-SNP read for them, so every piece stays connected.

Short reads only link SNPs fewer than `read_len[1]` apart, so `J` is zero far from the diagonal: a **band**. This matters twice later. It keeps the MPS simulator cheap (6.6) and the exact answer cheap (4.2).

| Knob | Default | Our "realistic" runs |
|---|---|---|
| `coverage` | 6 reads per SNP | 6 |
| `read_len` | 2–4 SNPs | 2–4 |
| `error` | 0 | 0.05 |
| `missing` | 0 | 0.10 |
| `gaps` | 0 | 0 |
| `seed` | 0 | puzzle k of every size uses seed k |

In [4]:
def make_instance(n, coverage=6.0, read_len=(2, 4), error=0.0, missing=0.0, gaps=0, seed=0):
    """Synthetic reads drawn from a hidden haplotype pair (truth, 1 - truth)."""
    rng = np.random.default_rng(seed)
    truth = rng.integers(0, 2, n)
    hi = min(read_len[1], n)
    lo = max(2, min(read_len[0], hi))
    cuts = sorted(rng.choice(np.arange(1, n), size=min(gaps, n - 1), replace=False).tolist()) if gaps else []
    reads = []

    def add(start, stop, drop=True):
        for c in cuts:                      # a read never crosses a gap
            if start < c < stop:
                stop = c
                break
        if stop - start < 2:
            return
        seg = (truth if rng.random() < 0.5 else 1 - truth)[start:stop].copy()   # copy A or copy B
        seg[rng.random(seg.size) < error] ^= 1                                  # sequencing errors
        if drop and missing:
            seg[rng.random(seg.size) < missing] = -1                            # letters not read
        row = np.full(n, -1)
        row[start:stop] = seg
        reads.append(row)

    for _ in range(max(1, math.ceil(coverage * n / ((lo + hi) / 2)))):
        length = int(rng.integers(lo, hi + 1))
        start = int(rng.integers(0, n - length + 1))
        add(start, start + length)
    # repair: every neighbouring pair inside a block gets at least one vote
    R = np.array(reads) if reads else np.full((0, n), -1)
    linked = ((R[:, :-1] >= 0) & (R[:, 1:] >= 0)).any(axis=0)
    for i in np.flatnonzero(~linked):
        if i + 1 not in cuts:
            add(int(i), int(i) + 2, drop=False)
    params = dict(coverage=coverage, read_len=f"{lo}-{hi}", error=error, missing=missing, gaps=gaps, seed=seed)
    return Instance(f"n{n}-s{seed}", np.array(reads), truth, params)


if not QGD_QUIET:
    demo = make_instance(10, error=0.05, missing=0.1, seed=1)
    show_reads(demo.reads, demo.truth)

read  1   . . 0 0 0 1 . . . .
read  2   . . . . 0 0 . . . .
read  3   . . 1 1 . . . . . .
read  4   . 1 1 1 . . . . . .
read  5   . . . 1 0 0 1 . . .
read  6   . . . . . 0 1 . 0 .
read  7   . . . . . . 0 0 0 1
read  8   . . . . 1 1 . 0 . .
read  9   . . . . . . . 0 1 1
read 10   1 0 . . . . . . . .
read 11   . 0 1 1 0 . . . . .
read 12   . . . 0 1 . . . . .
read 13   . 0 0 0 1 . . . . .
read 14   . . . 0 1 . . . . .
read 15   . . 1 1 0 . . . . .
read 16   . . . . 1 1 0 . . .
read 17   . . . . 0 0 . . . .
read 18   . . . . . 1 0 0 1 .
read 19   . . 1 1 0 0 . . . .
read 20   . . . . . . 1 1 0 .
truth     0 1 1 1 0 0 1 1 0 0   <- copy A (copy B is its mirror)


## 3 · From reads to a cost function

### 3.1 Counting votes: `A` and `D`

For every pair of SNPs (i, j), each read with letters at both is one vote. It votes **same** if the letters match and **opposite** if they don't. `A[i, j]` counts the same votes and `D[i, j]` the opposite ones. This is the notebook's `build_evidence`, rewritten with matrix products so it stays fast at 100+ SNPs:

* `one[r, i] = 1` when read r has letter 1 at SNP i, and `zero[r, i] = 1` when it has letter 0.
* `one.T @ one` counts, for every (i, j) at once, the reads with a 1 at both SNPs. Adding `zero.T @ zero` gives all "same" votes.
* `one.T @ zero + zero.T @ one` counts the reads with different letters, i.e. the "opposite" votes.
* `np.triu(…, 1)` keeps each pair once (i < j).

In [5]:
@register("evidence", "pairwise")
def pairwise_evidence(reads):
    """Vectorised build_evidence from the notebook: upper-triangular same / opposite vote counts."""
    R = np.asarray(reads)
    one = (R == 1).astype(float)    # float, not int: numpy only has the fast (BLAS) matrix product for floats;
    zero = (R == 0).astype(float)   # vote counts are small whole numbers, so floats hold them exactly
    same = (one.T @ one + zero.T @ zero).astype(np.int64)
    opposite = (one.T @ zero + zero.T @ one).astype(np.int64)
    return np.triu(same, 1), np.triu(opposite, 1)

### 3.2 Votes → `J` and `offset`, and why the formula is right

Take one pair with `a` same votes and `d` opposite votes. An answer either puts the two SNPs on the same copy ($z_i z_j = +1$) or on different copies ($z_i z_j = -1$). Its **cost** on this pair is the number of votes it disagrees with:

| The answer says | It disagrees with | Cost | $\frac{a+d}{2} + \frac{d-a}{2} z_i z_j$ |
|---|---|---|---|
| same copy (+1) | the `d` opposite votes | d | $\frac{a+d}{2} + \frac{d-a}{2} = d$ ✓ |
| different copies (−1) | the `a` same votes | a | $\frac{a+d}{2} - \frac{d-a}{2} = a$ ✓ |

Summing over all pairs gives $\text{cost}(x) = \text{offset} + \sum_{i<j} J_{ij} z_i z_j$ with **J = (D − A)/2** and **offset = Σ(A + D)/2**. The offset is the notebook's $c_0$.

* $J_{ij} < 0$ (more "same" votes) rewards putting i and j on the same copy. $J_{ij} > 0$ rewards splitting them.
* For a random answer every $z_i z_j$ averages to 0, so **offset = the average cost of a random answer**. The score in part 7 uses this.

Helpers below:
* `costs(X, J, offset)` scores many answers at once, one per row of X.
* `bits(k, n)` turns basis-state numbers into answers: bit i of k is SNP i + 1, which is qubit i.
* `decode_key` reads a Qiskit count key, which is written backwards (qubit 0 is the rightmost character).

In [6]:
def ising(A, D):
    A, D = np.triu(np.asarray(A, float), 1), np.triu(np.asarray(D, float), 1)
    return (D - A) / 2, float((A + D).sum() / 2)


def costs(X, J, offset):
    """Cost of each row of X (0/1, SNP 1 first)."""
    Z = 1.0 - 2.0 * np.asarray(X, float)
    return offset + ((Z @ J) * Z).sum(axis=1)


def bits(k, n):
    """Basis indices -> 0/1 rows; bit i of k is SNP i + 1 (Qiskit qubit i)."""
    return ((np.asarray(k, np.int64)[:, None] >> np.arange(n)) & 1).astype(np.int8)


def decode_key(key, n):
    """Qiskit count key -> bit array with SNP 1 first (same as the notebook)."""
    s = str(key).replace(" ", "")
    if len(s) != n or set(s) - {"0", "1"}:
        raise ValueError("Expected n measured binary bits.")
    return np.array([int(b) for b in s[::-1]], dtype=np.int8)


if not QGD_QUIET:
    A, D = pairwise_evidence(TOY_READS)
    J, offset = ising(A, D)
    print("toy: A[SNP1, SNP2] =", A[0, 1], " D[SNP1, SNP2] =", D[0, 1], " J[SNP1, SNP2] =", J[0, 1])
    print("offset (notebook c0) =", offset)
    for label, x in [("truth 01101", [0, 1, 1, 0, 1]), ("mirror 10010", [1, 0, 0, 1, 0]), ("all zeros", [0] * 5)]:
        print(f"cost({label}) = {costs([x], J, offset)[0]:g}")

toy: A[SNP1, SNP2] = 1  D[SNP1, SNP2] = 2  J[SNP1, SNP2] = 0.5
offset (notebook c0) = 12.5
cost(truth 01101) = 1
cost(mirror 10010) = 1
cost(all zeros) = 16


## 4 · The referee: the exact best cost

To grade a method we must know the best possible cost `opt`. There are two exact ways to get it.

### 4.1 Brute force with a doubling trick (up to about 24 SNPs)

`diag_energies(J)` lists $\sum J_{ij} z_i z_j$ for **all** $2^N$ answers at once, adding one SNP at a time:

* Suppose `E` holds the energies of every answer for the first q SNPs, and `f` holds SNP q's couplings to them, $\sum_{m<q} J_{mq} z_m$.
* Giving SNP q spin +1 adds `f`; spin −1 subtracts it. So the new list is `[E + f, E − f]`, twice as long. `f` is built by the same doubling.
* Total work is about $2 \cdot 2^N$ additions. Memory is $8 \cdot 2^N$ bytes: 134 MB at 24 SNPs.

The same list feeds the exact QAOA simulator (6.2), where the whole cost layer becomes one phase $e^{-i\gamma E}$.

In [7]:
def diag_energies(J):
    """sum_{i<j} J_ij z_i z_j for every basis index (bit i = qubit i), built by doubling in O(2^n)."""
    E = np.zeros(1)
    for q in range(len(J)):
        f = np.zeros(1)
        for m in range(q):
            f = np.concatenate([f + J[m, q], f - J[m, q]])
        E = np.concatenate([E + f, E - f])
    return E

### 4.2 Band dynamic programming (any N, as long as reads are short)

Brute force runs out of memory around 30 SNPs. But short reads make `J` a band of width `w` (`bandwidth`, 3 for reads of up to 4 SNPs), and that changes everything:

* Sweep the SNPs left to right. The cost added by SNP i depends only on the **last w SNPs**.
* So remember, for each of the $2^w$ settings of the last w SNPs, **the best cost so far** (`V`).
* One step: try both letters for SNP i, add its couplings to the window (`zb * lin`), then forget the oldest SNP by keeping the better of its two values (`min` over the pair).
* Cost: N steps × $2^w$ states, so **linear in N**. With w = 3 there are 8 states, and 1000 SNPs take milliseconds.

This is the classic Viterbi / transfer-matrix idea. The honest consequence: **short-read phasing is easy for a classical computer.** A quantum advantage would need long-range links (a wide band).

In [8]:
def bandwidth(J):
    i, j = np.nonzero(np.triu(J, 1))
    return int((j - i).max()) if len(i) else 0


def banded_exact(J, offset, max_band=16):
    """Exact optimum by dynamic programming over a sliding window of the last w SNPs, O(N * 2^w).

    Returns None if the band is wider than max_band.
    """
    n, w = len(J), bandwidth(J)
    if w == 0:
        return offset
    if w > max_band:
        return None
    s = np.arange(2 ** w)
    Zs = 1.0 - 2.0 * ((s[:, None] >> np.arange(w)) & 1)   # bit t of the state = SNP (i - 1 - t)
    V = np.zeros(2 ** w)
    for i in range(n):
        col = np.array([J[i - 1 - t, i] if i - 1 - t >= 0 else 0.0 for t in range(w)])
        lin = Zs @ col                                     # coupling of SNP i to each window state
        new = np.empty_like(V)
        for b, zb in ((0, 1.0), (1, -1.0)):
            cand = (V + zb * lin).reshape(2, -1).min(axis=0)   # drop the oldest bit
            new[(np.arange(2 ** (w - 1)) << 1) | b] = cand
        V = new
    return offset + float(V.min())


if not QGD_QUIET:
    demo = make_instance(14, error=0.05, missing=0.1, seed=2)
    J, offset = ising(*pairwise_evidence(demo.reads))
    print("14 SNPs, band", bandwidth(J), "| brute force:", offset + diag_energies(J).min(), "| band DP:", banded_exact(J, offset))
    big = make_instance(1000, error=0.05, missing=0.1, seed=2)
    J, offset = ising(*pairwise_evidence(big.reads))
    t = time.time()
    best = banded_exact(J, offset)
    print(f"1000 SNPs: exact optimum {best:g} in {time.time() - t:.3f} s (2^1000 answers)")

14 SNPs, band 3 | brute force: 6.0 | band DP: 6.0


1000 SNPs: exact optimum 490 in 0.036 s (2^1000 answers)


## 5 · Classical contestants

The built-in comparison uses **B = 2048 final tries**, the notebook's shot count. One try means:

| Method | One try = |
|---|---|
| random guessing | one random answer |
| greedy + local search | one cost evaluation (a scan of all N single flips = N tries) |
| simulated annealing | one proposed flip |
| QAOA | one final shot; training shots and post-processing checks are added to TTS99 |
| brute force | one of the $2^N$ answers (the referee; not held to the budget) |

Random guessing needs no function: the race draws B random rows (part 8). Our built-in trainers use zero training shots. For shot-trained plug-ins, B is still the final-sampling budget, **not an all-in cap**: training is charged separately in TTS99. Classical training time is reported separately, not priced as shots.

Both local methods use the **local field** trick. Let $h_i = \sum_j W_{ij} z_j$ with $W = J + J^T$. Flipping SNP i changes the cost by $\Delta E = -2 z_i h_i$, with no full recount. After a flip, `h` is updated with one row of W.

Both also run many independent **chains** side by side (`runs`, 64 by default) as numpy arrays. So one call measures a success rate, not a single lucky run.

In [9]:
def _sym(J):
    return J + J.T

### 5.1 Simulated annealing

At each step, every chain proposes flipping one random SNP.

* **Better or equal:** accept.
* **Worse by ΔE:** accept with probability $e^{-\Delta E / T}$.
* The temperature T falls geometrically over the budget. It starts **hot** (the largest total coupling of any SNP, so almost everything is accepted) and ends **cold** (5% of the smallest coupling, so only improvements are accepted).
* Early randomness escapes traps; late cold settles into a valley.

Per chain it returns the best cost, the best answer, the tries used, and `hit`: the try at which the chain first reached `target` (the optimum), or ∞ if it never did. `hit` feeds TTS99 (part 7).

In [10]:
def simulated_annealing(J, offset, budget, runs, rng, target=-math.inf):
    """Single-flip Metropolis on `runs` independent chains; each proposal is one evaluation."""
    n, W = len(J), _sym(J)
    nz = np.abs(W[W != 0])
    t_hot = float(np.abs(W).sum(axis=1).max()) or 1.0
    t_cold = 0.05 * float(nz.min()) if nz.size else 1.0
    Z = rng.choice([-1.0, 1.0], size=(runs, n))
    h = Z @ W
    E = offset + 0.5 * (Z * h).sum(axis=1)
    best, bestZ = E.copy(), Z.copy()
    hit = np.where(E <= target + 1e-9, 1.0, np.inf)
    ar = np.arange(runs)
    temps = np.geomspace(t_hot, t_cold, budget - 1) if budget > 1 else []
    for step, T in enumerate(temps):
        i = rng.integers(0, n, runs)                       # one random SNP per chain
        dE = -2.0 * Z[ar, i] * h[ar, i]
        ok = (dE <= 0) | (rng.random(runs) < np.exp(-np.maximum(dE, 0) / T))
        idx, ii = ar[ok], i[ok]
        z_old = Z[idx, ii].copy()
        Z[idx, ii] = -z_old
        h[idx] -= 2.0 * z_old[:, None] * W[ii]             # local fields after the flip
        E[idx] += dE[ok]
        better = E < best - 1e-12
        best[better], bestZ[better] = E[better], Z[better]
        hit[(E <= target + 1e-9) & np.isinf(hit)] = step + 2
    return best, ((1 - bestZ) / 2).astype(np.int8), np.full(runs, budget), hit

### 5.2 Greedy start + local search, with restarts

* **Greedy start** (costs 2N tries): visit the SNPs in a random order and give each the letter that agrees best with the SNPs already placed (the sign of its field). Ties are a coin flip.
* **Steepest descent** (N tries per scan): look at all N single flips and apply the best one if it lowers the cost.
* **Stuck** (no flip helps, a local minimum): restart from a fresh random greedy answer, while the budget lasts.

In [11]:
def _greedy(W, k, rng):
    """Randomised greedy: visit SNPs in random order, give each the letter that best fits so far."""
    n = len(W)
    order = np.argsort(rng.random((k, n)), axis=1)
    Z, ar = np.zeros((k, n)), np.arange(k)
    for t in range(n):
        i = order[:, t]
        fld = (W[i] * Z).sum(axis=1)
        tie = rng.choice([-1.0, 1.0], size=k)
        Z[ar, i] = np.where(fld > 0, -1.0, np.where(fld < 0, 1.0, tie))
    return Z


def greedy_local_search(J, offset, budget, runs, rng, target=-math.inf):
    """Greedy start + steepest 1-flip descent, restarting when stuck, until the budget is spent."""
    n, W = len(J), _sym(J)
    ar = np.arange(runs)
    Z = _greedy(W, runs, rng)
    used = np.full(runs, 2 * n)
    h = Z @ W
    E = offset + 0.5 * (Z * h).sum(axis=1)
    best, bestZ = E.copy(), Z.copy()
    hit = np.where(E <= target + 1e-9, used.astype(float), np.inf)
    while True:
        active = used + n <= budget
        if not active.any():
            break
        dE = -2.0 * Z * h                                  # cost change of every single flip
        j = dE.argmin(axis=1)
        m = dE[ar, j]
        used[active] += n
        imp = active & (m < -1e-12)
        if imp.any():
            idx, jj = ar[imp], j[imp]
            z_old = Z[idx, jj].copy()
            Z[idx, jj] = -z_old
            h[idx] -= 2.0 * z_old[:, None] * W[jj]
            E[imp] += m[imp]
        stuck = active & ~imp
        restart = stuck & (used + 2 * n <= budget)
        if restart.any():
            Z[restart] = _greedy(W, int(restart.sum()), rng)
            used[restart] += 2 * n
            h[restart] = Z[restart] @ W
            E[restart] = offset + 0.5 * (Z[restart] * h[restart]).sum(axis=1)
        newly = (E <= target + 1e-9) & np.isinf(hit)
        hit[newly] = used[newly]
        used[stuck & ~restart] = budget
        better = E < best - 1e-12
        best[better], bestZ[better] = E[better], Z[better]
    return best, ((1 - bestZ) / 2).astype(np.int8), np.minimum(used, budget), hit


if not QGD_QUIET:
    demo = make_instance(16, error=0.05, missing=0.1, seed=0)
    J, offset = ising(*pairwise_evidence(demo.reads))
    opt = banded_exact(J, offset)
    for name, solver in (("local search", greedy_local_search), ("annealing", simulated_annealing)):
        best, *_ = solver(J, offset, 2048, 64, np.random.default_rng(0), target=opt)
        print(f"{name:>12}: {np.mean(best <= opt + 1e-9):.0%} of 64 chains found the optimum ({opt:g}) within 2048 tries")

local search: 100% of 64 chains found the optimum (7) within 2048 tries
   annealing: 73% of 64 chains found the optimum (7) within 2048 tries


## 6 · QAOA

### 6.1 The circuit, identical to the baseline notebook's `make_qaoa`

* **H on every qubit:** start in an equal mix of all $2^N$ answers.
* **Cost layer (γ):** one `RZZ(2γ·J_ij / scale)` for each pair with $J_{ij} \ne 0$. It gives each answer a phase proportional to its cost.
* **Mixer (β):** `RX(2β)` on every qubit, so amplitude flows between answers that differ in one SNP. Phases plus mixing create interference that favours low-cost answers.
* Repeat cost layer + mixer p times (the **depth**), then measure.
* `scale = max(1, max|J|)` keeps rotation angles in a sensible range, so the same γ means about the same thing on different puzzles.

Registered as the `baseline` circuit. Carla's variants register next to it.

In [12]:
def qaoa_scale(J):
    return float(max(1, np.max(np.abs(J))))


@register("qaoa", "baseline")
def make_qaoa(J, p=1):
    """Same circuit as the notebook's make_qaoa: H, then per layer RZZ on active pairs and RX."""
    from qiskit import QuantumCircuit
    from qiskit.circuit import ParameterVector

    J = np.asarray(J, float)
    n = J.shape[0]
    scale = qaoa_scale(J)
    theta = ParameterVector("theta", 2 * p)
    gammas, betas = theta[:p], theta[p:]
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        for a in range(n):
            for b in range(a + 1, n):
                if J[a, b] != 0:
                    qc.rzz(2 * gammas[layer] * J[a, b] / scale, a, b)
        for q in range(n):
            qc.rx(2 * betas[layer], q)
    return qc, theta, scale

### 6.2 An exact simulator in 15 lines of numpy

The baseline circuit can be simulated without Qiskit:

* **State:** a vector ψ of $2^N$ complex amplitudes (16 bytes each). Start with all of them equal to $2^{-N/2}$, which is H on every qubit.
* **Cost layer = all RZZ gates at once.** They are all diagonal, so together they multiply amplitude k by $e^{-i (\gamma/\text{scale}) E_k}$, with `E` from `diag_energies`. That is one vector multiply.
* **Mixer:** `RX(2β)` on qubit q mixes each pair of amplitudes that differ only in bit q: $(a_0, a_1) \to (\cos\beta\, a_0 - i \sin\beta\, a_1,\; -i \sin\beta\, a_0 + \cos\beta\, a_1)$. `reshape(-1, 2, 2**q)` lines those pairs up.
* **Output:** exact probabilities $|\psi_k|^2$. So P(optimum) is exact, with no sampling noise.

Memory is $16 \cdot 2^N$ bytes: 1 GB at 26 SNPs, the limit we use (`sv_max`).

In [13]:
def qaoa_state(J, angles, E=None):
    """Exact ideal QAOA statevector for the baseline ansatz (numpy, no Qiskit), 16 * 2^N bytes."""
    n, p = len(J), len(angles) // 2
    scale = qaoa_scale(J)
    E = diag_energies(J) if E is None else E
    psi = np.full(2 ** n, 2 ** (-n / 2), dtype=np.complex128)
    for g, b in zip(angles[:p], angles[p:]):
        psi *= np.exp(-1j * (g / scale) * E)               # every RZZ of the layer at once
        c, s = math.cos(b), -1j * math.sin(b)
        for q in range(n):                                 # RX(2b) on qubit q
            v = psi.reshape(-1, 2, 2 ** q)
            a0 = v[:, 0, :].copy()
            v[:, 0, :] = c * a0 + s * v[:, 1, :]
            v[:, 1, :] = s * a0 + c * v[:, 1, :]
    return psi

### 6.3 Training p = 1 without shots: a formula instead of circuit simulation

The baseline notebook tunes (γ, β) with COBYLA, which re-runs the circuit many times. For p = 1 there is an exact formula for the expected cost (Ozaeta, van Dam & McMahon, 2020):

$$E(\gamma, \beta) = \text{offset} + \tfrac12 \sin 4\beta \cdot T_1(\gamma) - \tfrac12 \sin^2 2\beta \cdot T_2(\gamma)$$

* `p1_terms` computes $T_1, T_2$ from J using products of cosines over each pair's neighbours. That costs O(pairs · N) per γ: no circuit and no $2^N$ vector, so it works at 100+ SNPs.
* **For a fixed γ, the best β has a closed form.** Rewrite $E = \text{offset} - \frac{T_2}{4} + \frac{T_1}{2}\sin 4\beta + \frac{T_2}{4}\cos 4\beta$. A sum $a\sin + b\cos$ has minimum $-\sqrt{a^2 + b^2}$, so $\min_\beta E = \text{offset} - \frac{T_2}{4} - \sqrt{T_1^2/4 + T_2^2/16}$, reached at $4\beta = \text{atan2}(-2T_1, -T_2)$.
* So only γ is searched: a 256-point grid on (0, π], then a 1-D refinement (`minimize_scalar`) around the best grid point.
* `canonical_beta` folds β into (−π/4, π/4]. Shifting β by π/2 adds a global flip of every SNP, which maps each answer to its mirror with the same cost. So β and β + π/2 are equivalent. Folding makes angles from different puzzles comparable, which 6.4 needs.

Result: angles plus the predicted expected cost, with **zero shots**. Check 9.3 shows the prediction matches the simulator exactly; check 9.4 shows it is at least as good as the notebook's COBYLA result on the toy.

In [14]:
def p1_terms(J, gamma):
    """T1, T2 with E(gamma, beta) = offset + sin(4b)/2 * T1 - sin^2(2b)/2 * T2 for p = 1.

    gamma here is the physical phase (exp(-i gamma sum J Z Z)), i.e. notebook gamma / scale.
    """
    W, n = _sym(J), len(J)
    g = np.atleast_1d(np.asarray(gamma, float))[:, None]
    T1, T2 = np.zeros(g.shape[0]), np.zeros(g.shape[0])
    for u, v in zip(*np.nonzero(np.triu(J, 1))):
        rest = np.ones(n, bool)
        rest[[u, v]] = False
        Ju, Jv = W[u, rest], W[v, rest]
        Pu, Pv = np.cos(2 * g * Ju).prod(axis=1), np.cos(2 * g * Jv).prod(axis=1)
        Pp, Pm = np.cos(2 * g * (Ju + Jv)).prod(axis=1), np.cos(2 * g * (Ju - Jv)).prod(axis=1)
        T1 += J[u, v] * np.sin(2 * g[:, 0] * J[u, v]) * (Pu + Pv)
        T2 += J[u, v] * (Pp - Pm)
    return T1, T2


def p1_energy(J, offset, gamma_nb, beta):
    T1, T2 = p1_terms(J, np.asarray(gamma_nb, float) / qaoa_scale(J))
    return offset + np.sin(4 * beta) / 2 * T1 - np.sin(2 * beta) ** 2 / 2 * T2


def canonical_beta(beta):
    """Each beta only matters modulo pi/2 (a global bit flip leaves every cost unchanged): map to (-pi/4, pi/4]."""
    return -(np.mod(-np.asarray(beta, float) + np.pi / 4, np.pi / 2) - np.pi / 4)


def train_p1(J, offset, grid=256):
    """Best p = 1 angles (notebook units) from the closed form.

    Returns (angles [gamma, beta], predicted expected cost, classical evaluations). No shots needed.
    """
    from scipy.optimize import minimize_scalar

    scale = qaoa_scale(J)

    def best_beta(gnb):
        T1, T2 = p1_terms(J, np.atleast_1d(gnb) / scale)
        beta = 0.25 * np.arctan2(-2 * T1, -T2)
        return offset - np.sqrt(T1 ** 2 / 4 + T2 ** 2 / 16) - T2 / 4, canonical_beta(beta)

    gs = np.linspace(0, np.pi, grid + 1)[1:]
    vals, _ = best_beta(gs)
    k = int(vals.argmin())
    lo, hi = gs[max(k - 1, 0)], gs[min(k + 1, grid - 1)]
    res = minimize_scalar(lambda x: float(best_beta(x)[0][0]), bounds=(lo, hi), method="bounded")
    g = float(res.x) if res.fun < vals[k] else float(gs[k])
    e, b = best_beta(g)
    return np.array([g, float(b[0])]), float(e[0]), grid + int(res.nfev)

### 6.4 Training p = 2, 3: train once on small puzzles, reuse everywhere

For p ≥ 2 we do not use an analytic trainer. `train_transfer(p)` does this instead:

1. Make 9 small puzzles (3 each at N = 8, 10, 12) with the same generator settings as the race.
2. On each, start from the p = 1 formula. Then grow one layer at a time with **INTERP** (`interp`, Zhou et al. 2020), which stretches p angles onto p + 1 slots by linear interpolation. Polish each depth with COBYLA on the **exact** expected cost from the numpy simulator (no shots).
3. Take the **median** of each angle over the 9 puzzles. β is folded first, so medians make sense.

The same angles then run unchanged on every puzzle, up to 100 SNPs. This works because good QAOA angles **concentrate**: they barely depend on the specific puzzle (Brandão et al. 2018; visible in our runs too).

Cost: about 3,000–4,000 classical simulator evaluations, once per race, on vectors of at most 12 qubits. It takes seconds and **zero shots**.

In [15]:
def interp(angles):
    """INTERP initialisation (Zhou et al. 2020): angles for depth p -> starting angles for depth p + 1."""
    p = len(angles) // 2
    out = []
    for x in (np.r_[0.0, angles[:p], 0.0], np.r_[0.0, angles[p:], 0.0]):
        out += [(i - 1) / p * x[i - 1] + (p - i + 1) / p * x[i] for i in range(1, p + 2)]
    return np.array(out)


def train_transfer(p, pre_ns=(8, 10, 12), per_n=3, gen=None, seed=10_000):
    """Angles for p >= 2: train on small instances (exact, COBYLA), take the median, reuse at any N.

    Returns (angles, total classical evaluations spent on pre-training).
    """
    from scipy.optimize import minimize

    found, evals = [], 0
    for n in pre_ns:
        for k in range(per_n):
            inst = make_instance(n, seed=seed + 97 * n + k, **(gen or {}))
            J, off = ising(*pairwise_evidence(inst.reads))
            E = diag_energies(J)
            a, _, e1 = train_p1(J, off)
            evals += e1

            def f(x):
                return float(np.abs(qaoa_state(J, x, E)) ** 2 @ E) + off

            for depth in range(2, p + 1):
                r = minimize(f, interp(a), method="COBYLA", options={"maxiter": 150 * depth})
                evals += r.nfev
                a = np.r_[r.x[:depth], canonical_beta(r.x[depth:])]
            found.append(a)
    return np.median(np.array(found), axis=0), evals

### 6.5 Trainers (socket `trainer`)

A trainer picks the angles for one puzzle and reports how many **shots** it used. Those shots are added to the method's TTS99. Zero shots does not mean zero computation: `train_s` records classical training time separately.

* **`baseline`**: p = 1 uses the formula (6.3) on this puzzle. p ≥ 2 uses the transferred angles (6.4), computed once per race and kept in `ctx["cache"]`. **0 shots.** It only fits the baseline circuit, because the formula is specific to it.
* **`cobyla`**: for any circuit, e.g. Carla's. It runs COBYLA on the **sampled** mean cost, so each evaluation runs the circuit with `train_shots` shots on the chosen backend, and every shot is counted. It starts from the p = 1 formula angles grown with INTERP.

What `ctx` gives a trainer:

| Key | Meaning |
|---|---|
| `sample(angles, shots)` | run the circuit and get back `(shots, N)` answers |
| `E` | exact energies when available, else None |
| `gen` | the generator settings, to pre-train on similar puzzles |
| `cache` | shared by all puzzles of one race |
| `train_shots`, `train_iters` | budget hints for shot-based trainers |
| `log` | print progress |

In [ ]:
@register("trainer", "baseline")
def baseline_trainer(J, offset, p, builder, ctx):
    """Zero shots: p = 1 closed form on this instance; p >= 2 angles pre-trained once on small instances and reused."""
    if builder != "baseline":
        raise ValueError(f"the baseline trainer only fits the baseline circuit, not '{builder}'; use trainer='cobyla'")
    if p == 1:
        angles, e_pred, evals = train_p1(J, offset)
        return angles, 0, dict(train="closed-form", train_evals=evals, e_pred=e_pred)
    key = ("transfer", p)
    if key not in ctx["cache"]:
        ctx["cache"][key], evals = train_transfer(p, gen=ctx["gen"])
        ctx["log"](f"pre-trained p={p} angles on small instances ({evals} classical evaluations): {np.round(ctx['cache'][key], 4)}")
    return ctx["cache"][key], 0, dict(train="transfer", train_evals=0)


@register("trainer", "cobyla")
def cobyla_trainer(J, offset, p, builder, ctx):
    """Any circuit: COBYLA on the sampled mean cost; every evaluation runs the circuit and its shots are counted."""
    from scipy.optimize import minimize

    shots, used = ctx["train_shots"], 0
    if builder == 'baseline':
        start, _, _ = train_p1(J, offset)
        for _ in range(p - 1):
            start = interp(start)
    else:
        start = np.random.default_rng(ctx.get('seed', 0)).uniform(
            0, np.pi, ctx['parameter_count'])

    def f(x):
        nonlocal used
        used += shots
        return float(costs(ctx["sample"](x, shots), J, offset).mean())

    r = minimize(f, start, method="COBYLA", options={"maxiter": ctx["train_iters"] or 30 * p, "rhobeg": 0.1})
    return np.asarray(r.x, float), used, dict(train="cobyla", train_evals=int(r.nfev))

### 6.6 Simulators (socket `backend`)

| Name | How | Limit | Exact probabilities? |
|---|---|---|---|
| `numpy` | the simulator from 6.2 | baseline circuit only, ≤ 26 SNPs (1 GB) | yes |
| `aer_sv` | Qiskit Aer statevector | any circuit, about 30 SNPs (memory) | no, samples only |
| `aer_mps` | Qiskit Aer matrix product state | any circuit, 100+ SNPs if the band is narrow | no, samples only |

**Why MPS scales here.** An MPS stores the state as a chain of tensors, one per qubit. Their size depends on the **entanglement** across each cut. Our short-range couplings and shallow circuits keep it manageable; greater depth or long-range links can make MPS expensive too.

**The setting that made it fast.** An RZZ between qubits that are not neighbours in the chain, such as (i, i + 2), makes Aer swap qubits until the two are adjacent. `mps_swap_direction` chooses which way the swaps go. With our banded gates the default (`mps_swap_left`) was very slow and `mps_swap_right` was not: **99 s → 0.12 s** at 24 SNPs.

**Beyond 63 qubits.** Aer's device description stops at 63 qubits. For bigger circuits we skip the full transpile and only unroll to gates Aer runs natively (`AER_BASIS`).

The race falls back automatically: `numpy` when it can, `aer_sv` up to `sv_max`, then `aer_mps`.

In [17]:
@register("backend", "numpy")
def numpy_backend(job, shots, seed):
    """Exact ideal simulation of the baseline ansatz; also returns per-state probabilities."""
    if job["builder"] != "baseline":
        raise ValueError("numpy backend simulates the baseline ansatz only; use aer_sv or aer_mps.")
    probs = np.abs(qaoa_state(job["J"], job["angles"], job.get("E"))) ** 2
    probs /= probs.sum()
    k = np.random.default_rng(seed).choice(len(probs), size=shots, p=probs)
    return bits(k, len(job["J"])), probs


AER_OPTIONS = {"matrix_product_state": {"mps_swap_direction": "mps_swap_right"}}
AER_BASIS = ["h", "x", "sx", "rx", "ry", "rz", "u", "cx", "cz", "rzz", "rxx", "ryy", "measure"]


def _aer(method):
    def run(job, shots, seed):
        from qiskit import transpile
        from qiskit_aer import AerSimulator

        qc = job["circuit"].copy()
        qc.measure_all()
        n = job["circuit"].num_qubits
        sim = AerSimulator(method=method, **AER_OPTIONS.get(method, {}))
        # the simulator's target stops at 63 qubits; beyond that only unroll to gates Aer runs natively
        tc = transpile(qc, sim) if n <= sim.target.num_qubits else transpile(qc, basis_gates=AER_BASIS, optimization_level=0)
        counts = sim.run(tc, shots=shots, seed_simulator=seed).result().get_counts()
        return np.array([decode_key(k, n) for k, c in counts.items() for _ in range(c)]), None
    run.__doc__ = f"Qiskit Aer, method={method}; works with any registered QAOA builder."
    return run


REGISTRY["backend"]["aer_sv"] = _aer("statevector")
REGISTRY["backend"]["aer_mps"] = _aer("matrix_product_state")

### 6.7 Post-processing (socket `postprocess`)

A post-processor sees the QAOA samples before grading. It may change them, for example by polishing each one with a few local flips. Two rules:

* Keep **one row per shot**, because the success rate is measured over these rows.
* Report **every extra cost evaluation**. The race spreads them over the shots, so each shot costs `1 + extra / B` tries in TTS99.

Built in: `none`. Alex's ideas plug in here.

In [18]:
@register("postprocess", "none")
def no_postprocess(samples, J, offset):
    return samples, 0

## 7 · Grading

| Metric | Question | How |
|---|---|---|
| `P_budget` | Did it find the optimum within B = 2048 tries? | samplers: $1 - (1-p)^B$, where p = chance that one shot is optimal. Local search / annealing: fraction of the 64 chains that hit it |
| `tts99` | How many tries to be 99% sure of seeing the optimum? | $\log(0.01) / \log(1-p)$ tries for success chance p per try; **lower is better** |
| `score` | How good was the best answer found? | $(\text{offset} - \text{best}) / (\text{offset} - \text{opt})$: 1 = optimum, 0 = no better than an average random guess |
| `switch_err` | How far from the hidden truth? | the number of places where the answer jumps between copy A and copy B, counted only where a read links the two SNPs |

Also recorded: `truth_dist` (wrong letters, up to the mirror) and `n_opt` (how many answers tie for the optimum, always ≥ 2 because of the mirror).

**TTS99 example.** If one shot is optimal with p = 1%, then $\log(0.01)/\log(0.99) ≈ 458$ shots give 99% confidence. If p ≥ 99%, one try is enough.

**TTS99 for chains** (`tts99_runs`). A chain stopped after b tries succeeds with chance s(b) = the fraction of chains with `hit` ≤ b. Repeating such chains costs $b \cdot \log(0.01)/\log(1 - s(b))$ tries. We take the best stopping point b.

**Why switch errors.** Wrong letters punish one swapped block by its length. Switch errors count it once, which is the standard phasing metric. A boundary that no read crosses cannot be phased by anyone, so it is skipped (`linked_boundaries`). That function marks the boundaries from i to j − 1 for every linked pair (i, j), using a difference array and a running sum.

In [19]:
def tts99(p, unit):
    """Expected cost (in `unit`s) to reach the optimum with 99% confidence, given success prob p."""
    if p <= 0:
        return math.inf
    if p >= 0.99:
        return float(unit)
    return unit * math.log(0.01) / math.log(1 - p)


def tts99_runs(hit):
    """Best TTS99 over stopping points b: runs stopped after b evaluations succeed with s(b) = P(hit <= b)."""
    h = np.sort(np.asarray(hit, float))
    best = math.inf
    for k, b in enumerate(h[np.isfinite(h)]):
        best = min(best, tts99((k + 1) / len(h), b))
    return best


def truth_distance(x, truth):
    if truth is None:
        return np.nan
    d = int(np.sum(np.asarray(x) != truth))
    return min(d, len(truth) - d)


def linked_boundaries(A, D):
    """Boundary i sits between SNP i+1 and i+2; it is phased only if some read links SNPs across it."""
    n = len(A)
    a, b = np.nonzero(np.triu(np.asarray(A) + np.asarray(D), 1))
    diff = np.zeros(n + 1, dtype=np.int64)
    np.add.at(diff, a, 1)
    np.add.at(diff, b, -1)
    return np.cumsum(diff)[: n - 1] > 0


def switch_errors(x, truth, linked):
    """Places where x switches between truth and its complement; unlinked boundaries are skipped."""
    if truth is None:
        return np.nan
    h = np.asarray(x, dtype=np.int64) ^ np.asarray(truth, dtype=np.int64)
    return int(np.sum((h[1:] != h[:-1]) & linked))


if not QGD_QUIET:
    truth = np.array([0, 1, 1, 0, 1, 0, 0, 1])
    answer = np.r_[truth[:4], 1 - truth[4:]]        # second half phased the other way round
    print("truth ", truth, "\nanswer", answer)
    print("wrong letters:", truth_distance(answer, truth), "| switch errors:", switch_errors(answer, truth, np.ones(7, bool)))
    print("TTS99 at p = 1% per shot:", round(tts99(0.01, 1)), "shots")

truth  [0 1 1 0 1 0 0 1] 
answer [0 1 1 0 0 1 1 0]
wrong letters: 4 | switch errors: 1
TTS99 at p = 1% per shot: 458 shots


## 8 · The race

### 8.1 Settings: `Config`

| Field | Default | Meaning |
|---|---|---|
| `methods` | brute, random, local, sa, qaoa | who races |
| `shots` | 2048 | the budget B for every method |
| `runs` | 64 | chains for local search / annealing |
| `ps` | (1,) | QAOA depths to try |
| `evidence`, `qaoa`, `trainer`, `postprocess`, `backend` | pairwise, baseline, auto, none, numpy | which plug-in sits in each socket (`trainer="auto"` = `baseline` for the baseline circuit, `cobyla` for any other) |
| `brute_max` | 24 | largest N for brute-force enumeration |
| `sv_max` | 26 | largest N for statevector simulation (1 GB) |
| `train_shots`, `train_iters` | 256, 0 (= 30·p) | budget for shot-based trainers |
| `gen` | {} | `make_instance` settings, e.g. `dict(error=0.05, missing=0.1)` |
| `seed` | 0 | puzzle k of each size uses seed `seed + k` |

In [20]:
@dataclass
class Config:
    methods: tuple = ("brute", "random", "local", "sa", "qaoa")
    shots: int = 2048            # budget per method: samples (random, QAOA) or evaluations (local, SA)
    runs: int = 64               # independent runs used to estimate success for local search / SA
    ps: tuple = (1,)
    backend: str = "numpy"
    evidence: str = "pairwise"
    qaoa: str = "baseline"
    postprocess: str = "none"
    brute_max: int = 24          # exact enumeration limit for the "brute" row (2^24 costs)
    sv_max: int = 26             # numpy statevector limit (16 * 2^26 B = 1.07 GB)
    seed: int = 0
    trainer: str = "auto"        # auto = "baseline" for the baseline circuit, "cobyla" for any other
    train_shots: int = 256       # shots per training evaluation (shot-based trainers)
    train_iters: int = 0         # optimiser iterations for shot-based trainers; 0 = 30 * p
    gen: dict = field(default_factory=dict)   # make_instance settings; trainers may pre-train on similar puzzles

    def trainer_for(self):
        return self.trainer if self.trainer != "auto" else ("baseline" if self.qaoa == "baseline" else "cobyla")

### 8.2 One puzzle: `run_instance`

For one puzzle it returns one result row per method (and per depth p for QAOA):

1. **Cost.** Build `A`, `D` with the chosen evidence socket, then `J`, `offset`, and the linked boundaries.
2. **Referee.** Enumerate all answers when N ≤ `brute_max` (or when the numpy simulator needs the energies anyway). Otherwise use the band DP. If the band is too wide, use the best known cost: annealing with 200× the budget, or the truth's cost if lower. The `ref` column says which was used.
3. **Brute force** row: always exact, costs $2^N$ tries.
4. **Random guessing:** B random answers. Its exact chance per try is n_opt / $2^N$.
5. **Local search and annealing:** 64 chains each, all with the optimum as `target`, so `hit` gives TTS99.
6. **QAOA, for each depth p:** pick the backend (falling back from numpy when needed), train with the trainer socket, sample B shots, apply post-processing, and grade. With the numpy simulator, P(optimum) is the exact sum of probabilities over all optimal answers. **If anything fails** (out of memory, a simulator limit, a plug-in bug) the error goes into a `fail` column and the race continues.

Every method on a puzzle shares the puzzle's random generator, seeded from the size and the puzzle name, so reruns give identical numbers.

In [ ]:
def run_instance(inst, cfg, cache=None, log=print):
    rng = np.random.default_rng(cfg.seed + 7919 * inst.n + zlib.crc32(inst.name.encode()) % 1000)
    A, D = REGISTRY["evidence"][cfg.evidence](inst.reads)
    J, offset = ising(A, D)
    n, B = inst.n, cfg.shots
    linked = linked_boundaries(A, D)
    base = dict(instance=inst.name, n=n, pairs=int(np.count_nonzero(J)), band=bandwidth(J), offset=offset,
                linked=int(linked.sum()), evidence=cfg.evidence, postprocess=cfg.postprocess, **inst.params)
    rows = []

    # -- referee: the exact optimum (or the best known one)
    t = time.time()
    numpy_qaoa = "qaoa" in cfg.methods and cfg.backend == "numpy" and cfg.qaoa == "baseline" and n <= cfg.sv_max
    E = diag_energies(J) if (n <= cfg.brute_max or numpy_qaoa) else None
    n_opt = None
    if E is not None:
        opt, opt_exact, ref = offset + float(E.min()), True, "enumeration"
        n_opt = int(np.isclose(E, E.min()).sum())
    else:
        opt = banded_exact(J, offset)
        opt_exact, ref = opt is not None, "band-dp"
        if opt is None:
            b, _, _, _ = simulated_annealing(J, offset, 200 * B, 16, rng)
            truth_cost = costs(inst.truth[None], J, offset)[0] if inst.truth is not None else math.inf
            opt, ref = float(min(b.min(), truth_cost)), "best-known"
    t_ref = time.time() - t
    gap = max(offset - opt, 1e-12)   # offset = average cost of a uniformly random answer
    is_opt = lambda c: np.asarray(c) <= opt + 1e-9

    def vs_truth(x):
        sw = switch_errors(x, inst.truth, linked)
        return dict(truth_dist=truth_distance(x, inst.truth), switch_err=sw, switch_rate=sw / max(int(linked.sum()), 1))

    def row(method, **kw):
        r = dict(base, method=method, opt=opt, opt_exact=opt_exact, ref=ref, n_opt=n_opt, budget=B, **kw)
        if "best" in r:
            r["score"] = (offset - r["best"]) / gap
        rows.append(r)

    if "brute" in cfg.methods and n <= cfg.brute_max:
        x = bits([int(np.argmin(E))], n)[0]
        row("brute", p_success=1.0, P_budget=1.0, tts99=float(2 ** n), evals=2 ** n, best=opt,
            wall_s=t_ref, **vs_truth(x))

    # -- samplers (random, QAOA): post-process, then grade the B rows
    def sampler_row(method, X, p_exact, wall, train_shots=0, **kw):
        X, pp_evals = REGISTRY["postprocess"][cfg.postprocess](X, J, offset)
        c = costs(X, J, offset)
        p = p_exact if (p_exact is not None and cfg.postprocess == "none") else float(is_opt(c).mean())
        x = X[int(np.argmin(c))]
        row(method, p_success=p, P_budget=1 - (1 - p) ** B, tts99=tts99(p, 1 + pp_evals / B) + train_shots,
            evals=B + pp_evals, best=float(c.min()), mean=float(c.mean()), found=bool(is_opt(c).any()),
            wall_s=wall, train_shots=train_shots, **vs_truth(x), **kw)

    if "random" in cfg.methods:
        t = time.time()
        X = rng.integers(0, 2, size=(B, n), dtype=np.int8)
        sampler_row("random", X, n_opt / 2 ** n if n_opt else None, time.time() - t)

    # -- local search and annealing: 64 chains each
    for name, solver in (("local", greedy_local_search), ("sa", simulated_annealing)):
        if name in cfg.methods:
            t = time.time()
            best, X, used, hit = solver(J, offset, B, cfg.runs, rng, target=opt)
            s = float(is_opt(best).mean())
            row(name, p_success=s, P_budget=s, tts99=tts99_runs(hit), evals=int(used.mean()), best=float(best.min()),
                mean=float(best.mean()), found=bool(s > 0),
                wall_s=time.time() - t, **vs_truth(X[int(np.argmin(best))]))

    # -- QAOA at each depth p
    if "qaoa" in cfg.methods:
        backend = cfg.backend
        if backend == "numpy" and (cfg.qaoa != "baseline" or n > cfg.sv_max):
            backend = "aer_sv" if n <= cfg.sv_max else "aer_mps"
        trainer = cfg.trainer_for()
        for p in cfg.ps:
            job = dict(J=J, E=E, builder=cfg.qaoa)
            if backend != "numpy":
                job["template"] = REGISTRY["qaoa"][cfg.qaoa](J, p)

            def run_job(angles, shots, seed, job=job):
                j = dict(job, angles=np.asarray(angles, float))
                if "template" in job:
                    qc, theta, _ = job["template"]
                    if len(j["angles"]) != len(theta):
                        raise ValueError("Trainer returned the wrong number of angles.")
                    j["circuit"] = qc.assign_parameters(dict(zip(theta, j["angles"])))
                return REGISTRY["backend"][backend](j, shots, seed)

            ctx = dict(sample=lambda a, s, run_job=run_job: run_job(a, s, int(rng.integers(2 ** 31)))[0],
                       E=E, gen=cfg.gen, cache=cache if cache is not None else {}, log=log,
                       train_shots=cfg.train_shots, train_iters=cfg.train_iters,
                       parameter_count=len(job["template"][1]) if "template" in job else 2 * p,
                       seed=cfg.seed)
            t = time.time()
            try:
                angles, train_shots, *extra = REGISTRY["trainer"][trainer](J, offset, p, cfg.qaoa, ctx)
                t_train = time.time() - t
                t = time.time()
                X, probs = run_job(angles, B, cfg.seed + n)
            except Exception as exc:   # out of memory, simulator limits or a plug-in failure: record and continue
                row(f"qaoa-p{p}", backend=backend, trainer=trainer, fail=repr(exc)[:200], wall_s=time.time() - t)
                log(f"  qaoa p={p} on {backend} failed: {exc!r}"[:200])
                continue
            p_exact = float(probs[np.isclose(E, opt - offset)].sum()) if probs is not None else None
            info = {"train_evals": 0, "e_pred": np.nan, **(extra[0] if extra else {})}
            info.setdefault("train", trainer)
            sampler_row(f"qaoa-p{p}", X, p_exact, time.time() - t, train_shots=int(train_shots), backend=backend,
                        builder=cfg.qaoa, trainer=trainer, train_s=t_train,
                        angles=" ".join(f"{a:.4f}" for a in angles),
                        mem_mb=16 * 2 ** n / 1e6 if backend in ("numpy", "aer_sv") else np.nan, **info)
    return rows

### 8.3 Many puzzles, a summary and a plot

* `run_bench(ns, instances, cfg)` runs `instances` puzzles for each size in `ns` (`"toy"` = the notebook's puzzle) and returns all rows. One `cache` is shared, so the transferred angles are trained once per race.
* `summarize(df)` gives the **median** per size and method, including ∞ for zero estimated success. Crashed rows have missing metrics, which pandas excludes: always inspect the `fail` column before reporting a summary.
* `plot(df, out=None)` draws the five standard panels. With `out` it also saves `scaling.png` there.

In [22]:
def run_bench(ns, instances=3, cfg=None, gen=None, log=print):
    cfg = cfg or Config()
    if gen is not None:
        cfg = replace(cfg, gen=gen)
    cache = {}   # shared by trainers across instances, e.g. the baseline trainer's transferred angles
    rows = []
    for n in ns:
        t, start = time.time(), len(rows)
        for k in range(1 if n == "toy" else instances):
            inst = toy_instance() if n == "toy" else make_instance(int(n), seed=cfg.seed + k, **cfg.gen)
            rows += run_instance(inst, cfg, cache, log)
        mean = {}
        for r in rows[start:]:
            if "P_budget" in r:
                mean.setdefault(r["method"], []).append(r["P_budget"])
        log(f"N={n}: {time.time() - t:.1f}s  P(optimum within budget): "
            + ", ".join(f"{m}={np.mean(v):.2f}" for m, v in mean.items()))
    return rows


def summarize(df):
    cols = [c for c in ("P_budget", "tts99", "score", "switch_err", "wall_s") if c in df]
    return df.groupby(["n", "method"])[cols].median().round(4)


def plot(df, out=None):
    import matplotlib.pyplot as plt

    # median over instances *including* failures (inf), so a lone lucky instance does not make a point
    g = df.groupby(["method", "n"]).median(numeric_only=True).reset_index().replace([np.inf], np.nan)
    fig, axes = plt.subplots(2, 3, figsize=(18, 8.5))
    axes, legend_ax = axes.ravel()[:5], axes.ravel()[5]
    colors = dict(zip(sorted(g["method"].unique()), plt.rcParams["axes.prop_cycle"].by_key()["color"] * 3))
    panels = [("P_budget", "P(find the optimum within the budget)", dict(ylim=(-0.03, 1.03))),
              ("score", "Solution quality: 1 = optimum, 0 = random guess", dict(ylim=(None, 1.02))),
              ("switch_rate", "Switch-error rate vs the truth (0 = perfect)", {}),
              ("tts99", "Shots / evaluations to reach the optimum with 99%", dict(yscale="log")),
              ("wall_s", "Wall time on this machine (s; QAOA = classical simulation)", dict(yscale="log"))]
    for m, d in g.groupby("method"):
        for ax, (col, _, _) in zip(axes, panels):
            if col in d:
                ax.plot(d["n"], d[col], marker="o", label=m, color=colors[m])
    for ax, (_, title, kw) in zip(axes, panels):
        ax.set(title=title, xlabel="N SNPs", **kw)
        ax.grid(alpha=0.3)
    legend_ax.axis("off")
    legend_ax.legend(*axes[0].get_legend_handles_labels(), loc="center", fontsize=12, title="method")
    fig.tight_layout()
    if out:
        os.makedirs(out, exist_ok=True)
        fig.savefig(os.path.join(out, "scaling.png"), dpi=120)
    plt.close(fig)
    return fig

## 9 · Checks

Each check proves one claim made above. They run when this notebook is opened on its own (about 10 s) and are skipped when another notebook loads it with `QGD_QUIET = True`.

| Check | Proves |
|---|---|
| 9.1 | votes, costs and the two optima on the toy match the baseline notebook |
| 9.2 | the doubling trick and the band DP agree with plain enumeration, also with errors, missing letters and gaps |
| 9.3 | the numpy simulator matches Qiskit's statevector, and the p = 1 formula matches the simulator |
| 9.4 | on the toy, the p = 1 formula is at least as good as the notebook's COBYLA (expected cost 7.798) |
| 9.5 | p = 2 angles trained on small puzzles beat p = 1 on an unseen 13-SNP puzzle |
| 9.6 | Aer MPS samples match the exact distribution, and it runs at 70 qubits |
| 9.7 | switch errors count one swapped block once and skip unlinked boundaries |
| 9.8 | error-free reads make the truth optimal with cost 0 |

In [23]:
def check_all():
    # 9.1 toy = baseline notebook
    A, D = pairwise_evidence(TOY_READS)
    assert (A[0, 1], D[0, 1], A[1, 2], D[2, 3], A[1, 4] + D[1, 4]) == (1, 2, 4, 4, 0)
    J, off = ising(A, D)
    E = diag_energies(J)
    best = np.flatnonzero(np.isclose(E, E.min()))
    assert off + E.min() == 1
    assert sorted("".join(map(str, x)) for x in bits(best, 5)) == ["01101", "10010"]
    print("9.1 toy matches the notebook: optimum cost 1 at 01101 and its mirror 10010")

    # 9.2 doubling trick and band DP = enumeration
    for seed, gaps in [(1, 0), (2, 1), (3, 2)]:
        inst = make_instance(11, error=0.1, missing=0.1, gaps=gaps, read_len=(2, 5), seed=seed)
        J, off = ising(*pairwise_evidence(inst.reads))
        c = costs(bits(np.arange(2 ** 11), 11), J, off)
        assert np.allclose(off + diag_energies(J), c)
        assert np.isclose(banded_exact(J, off), c.min())
    print("9.2 diag_energies and banded_exact agree with enumeration (errors, missing letters, gaps)")

    # 9.3 numpy simulator = Qiskit; p = 1 formula = simulator
    from qiskit.quantum_info import Statevector
    inst = make_instance(7, error=0.15, read_len=(2, 5), seed=5)
    J, off = ising(*pairwise_evidence(inst.reads))
    E = diag_energies(J)
    for angles in ([0.7, 0.3], [2.1, 1.2], [0.4, 1.1, 0.9, 0.2]):
        qc, theta, _ = make_qaoa(J, len(angles) // 2)
        ref = Statevector(qc.assign_parameters(dict(zip(theta, angles)))).probabilities()
        probs = np.abs(qaoa_state(J, np.array(angles), E)) ** 2
        assert np.allclose(probs, ref, atol=1e-10)
        if len(angles) == 2:
            assert np.isclose(p1_energy(J, off, angles[0], angles[1])[0], off + probs @ E)
    print("9.3 qaoa_state matches Qiskit Statevector; the p = 1 formula matches the simulated energy")

    # 9.4 formula >= notebook COBYLA on the toy
    J, off = ising(*pairwise_evidence(TOY_READS))
    angles, e_pred, _ = train_p1(J, off)
    assert np.isclose(e_pred, off + np.abs(qaoa_state(J, angles)) ** 2 @ diag_energies(J))
    assert e_pred <= 7.798 + 1e-6
    print(f"9.4 toy p = 1: formula expected cost {e_pred:.3f} <= notebook COBYLA 7.798, zero shots")

    # 9.5 transferred p = 2 beats p = 1 on an unseen puzzle
    angles2, _ = train_transfer(2, pre_ns=(8, 10), per_n=2)
    inst = make_instance(13, seed=123)
    J, off = ising(*pairwise_evidence(inst.reads))
    E = diag_energies(J)
    _, e1, _ = train_p1(J, off)
    e2 = off + np.abs(qaoa_state(J, angles2, E)) ** 2 @ E
    assert e2 < e1
    print(f"9.5 unseen 13-SNP puzzle: transferred p = 2 expected cost {e2:.3f} < p = 1 {e1:.3f}")

    # 9.6 Aer MPS: right distribution, and past 63 qubits
    for n, shots in ((9, 4000), (70, 16)):
        inst = make_instance(n, error=0.05, seed=6)
        J, off = ising(*pairwise_evidence(inst.reads))
        angles = np.array([0.4, 0.5, -0.6, -0.3])
        qc, theta, _ = make_qaoa(J, 2)
        job = dict(J=J, angles=angles, builder="baseline", circuit=qc.assign_parameters(dict(zip(theta, angles))))
        X, _ = REGISTRY["backend"]["aer_mps"](job, shots, 1)
        assert X.shape == (shots, n)
        if n <= 12:
            probs = np.abs(qaoa_state(J, angles)) ** 2
            seen = np.bincount(X @ (1 << np.arange(n)), minlength=2 ** n) / shots
            assert 0.5 * np.abs(seen - probs).sum() < 0.1
    print("9.6 Aer MPS samples match the exact distribution (9 qubits) and it runs at 70 qubits")

    # 9.7 switch errors
    truth = np.array([0, 1, 1, 0, 1, 0, 0, 1])
    flipped = truth.copy()
    flipped[4:] ^= 1
    linked = np.ones(7, dtype=bool)
    assert truth_distance(flipped, truth) == 4 and switch_errors(flipped, truth, linked) == 1
    assert switch_errors(1 - truth, truth, linked) == 0
    linked[3] = False
    assert switch_errors(flipped, truth, linked) == 0
    A, D = pairwise_evidence(make_instance(12, gaps=1, seed=2).reads)
    assert linked_boundaries(A, D).sum() == 10
    print("9.7 one swapped block = 1 switch error; unlinked boundaries are skipped")

    # 9.8 error-free reads: the truth is optimal with cost 0
    inst = make_instance(12, seed=4)
    J, off = ising(*pairwise_evidence(inst.reads))
    assert costs(inst.truth[None], J, off)[0] == 0 == banded_exact(J, off)
    print("9.8 error-free reads: the hidden truth has cost 0 and is optimal")


if not QGD_QUIET:
    t = time.time()
    check_all()
    print(f"all checks passed in {time.time() - t:.1f} s")

9.1 toy matches the notebook: optimum cost 1 at 01101 and its mirror 10010
9.2 diag_energies and banded_exact agree with enumeration (errors, missing letters, gaps)


9.3 qaoa_state matches Qiskit Statevector; the p = 1 formula matches the simulated energy


9.4 toy p = 1: formula expected cost 5.833 <= notebook COBYLA 7.798, zero shots


9.5 unseen 13-SNP puzzle: transferred p = 2 expected cost 9.416 < p = 1 14.617


9.6 Aer MPS samples match the exact distribution (9 qubits) and it runs at 70 qubits
9.7 one swapped block = 1 switch error; unlinked boundaries are skipped
9.8 error-free reads: the hidden truth has cost 0 and is optimal
all checks passed in 19.8 s
